In [ ]:
import json
import os

import torch
from sae_lens import SAE, ActivationsStore, LanguageModelSAERunnerConfig, SAEConfig
from sae_lens.evals import EvalConfig, run_evals
from sae_lens.sae import SAE_CFG_PATH, SAE_WEIGHTS_PATH
from safetensors.torch import save_file
from transformer_lens import HookedTransformer

from sae import Sae, SaeConfig, TrainConfig

In [ ]:
layer_index = 5
sae_lens_path = (
    "/Users/belerico/Desktop/ICLR 2025/checkpoints-baselines/sae_lens/layers.{}".format(
        layer_index
    )
)
eleuther_sae = Sae.load_from_disk(
    "/Users/belerico/Desktop/ICLR 2025/checkpoints-baselines/layers.{}".format(layer_index)
)
eleuther_sae_cfg = TrainConfig.load_json(
    "/Users/belerico/Desktop/ICLR 2025/checkpoints-baselines/config.json"
)
eleuther_sae_scale_factors = torch.load(
    "/Users/belerico/Desktop/ICLR 2025/checkpoints-baselines/scaling_factors.pt"
)

In [ ]:
[name for name, _ in list(eleuther_sae.named_parameters())]

In [ ]:
print(eleuther_sae_cfg)

In [ ]:
eleuther_sae_scale_factors

In [ ]:
def to_sae_lens(
    sae_cfg: SaeConfig,
    sae: Sae,
    model_name: str,
    dataset_path: str,
    output_path: str,
    max_seq_len: int = 1024,
    hook_name: str = "blocks.0.hook_resid_post",
    hook_layer: int = 0,
    dtype: str = "float32",
    device: str = "cuda",
):
    if sae_cfg.k > 0:
        architecture = "topk"
    elif sae_cfg.k <= 0:
        if sae_cfg.jumprelu:
            architecture = "jumprelu"
        else:
            architecture = "standard"
    else:
        raise ValueError("Invalid SAE architecture")
    d_in = sae.d_in
    d_sae = sae.num_latents
    activation_fn_str = "relu" if sae_cfg.k <= 0 else "topk"
    apply_b_dec_to_input = True
    finetuning_scaling_factor = False

    # dataset it was trained on details.
    context_size = max_seq_len
    model_name = model_name
    hook_name = hook_name
    hook_layer = hook_layer
    hook_head_index = None
    prepend_bos = False
    dataset_path = dataset_path
    dataset_trust_remote_code = True
    normalize_activations = "none"

    # misc
    dtype = dtype
    device = device
    sae_lens_cfg =  SAEConfig(
        architecture=architecture,
        d_in=d_in,
        d_sae=d_sae,
        activation_fn_str=activation_fn_str,
        apply_b_dec_to_input=apply_b_dec_to_input,
        finetuning_scaling_factor=finetuning_scaling_factor,
        context_size=context_size,
        model_name=model_name,
        hook_name=hook_name,
        hook_layer=hook_layer,
        hook_head_index=hook_head_index,
        prepend_bos=prepend_bos,
        dataset_path=dataset_path,
        dataset_trust_remote_code=dataset_trust_remote_code,
        normalize_activations=normalize_activations,
        dtype=dtype,
        device=device,
        sae_lens_training_version="4.4.5",
    )
    os.makedirs(output_path, exist_ok=True)
    with open(f"{output_path}/{SAE_CFG_PATH}", "w") as f:
            json.dump(sae_lens_cfg.to_dict(), f)
    state_dict = sae.state_dict()
    state_dict = {k: v.cpu() for k, v in state_dict.items()}
    state_dict["W_enc"] = state_dict.pop("encoder.weight").T.contiguous()
    state_dict["b_enc"] = state_dict.pop("encoder.bias")
    if "log_threshold" in state_dict:
        state_dict["threshold"] = torch.exp(state_dict.pop("log_threshold"))
    for k, p in list(state_dict.items()):
        print(k, p.shape)
    save_file(state_dict, f"{output_path}/{SAE_WEIGHTS_PATH}")



In [ ]:
to_sae_lens(
    eleuther_sae.cfg,
    eleuther_sae,
    "EleutherAI/pythia-160m-deduped",
    "NeelNanda/pile-small-tokenized-2b",
    "./sae_lens/pythia-160m-deduped-layers.{}".format(layer_index),
    hook_name="blocks.{}.hook_resid_post".format(layer_index),
    hook_layer=layer_index,
    device="cpu"
)

In [ ]:
sae_lens_sae = SAE.load_from_pretrained(
    "./sae_lens/pythia-160m-deduped-layers.{}".format(layer_index)
)
sae_lens_sae.fold_activation_norm_scaling_factor(
    eleuther_sae_scale_factors["layers.{}".format(layer_index)]
)
# sae_lens_sae.activation_fn = nn.Identity()

In [ ]:
model = HookedTransformer.from_pretrained("EleutherAI/pythia-160m-deduped").to("mps")

# Configure the runner for the current SAE model
cfg = LanguageModelSAERunnerConfig(
    model_name=sae_lens_sae.cfg.model_name,
    hook_name=sae_lens_sae.cfg.hook_name,
    hook_layer=int(sae_lens_sae.cfg.hook_layer),
    dataset_path="NeelNanda/pile-small-tokenized-2b",
    is_dataset_tokenized=False,
    context_size=1024,
    streaming=True,
    # Activation Store Parameters
    n_batches_in_buffer=4,
    verbose=False,
    device="mps",
    prepend_bos=sae_lens_sae.cfg.prepend_bos,
    architecture=sae_lens_sae.cfg.architecture,
)

eval_cfg: EvalConfig = EvalConfig(
    batch_size_prompts=4,
    # Reconstruction metrics
    n_eval_reconstruction_batches=4,
    compute_kl=False,  # Disable KL divergence computation
    compute_ce_loss=True,  # Enable cross-entropy loss computation
    # Sparsity and variance metrics
    n_eval_sparsity_variance_batches=4,
    compute_l2_norms=True,  # Compute L2 norms
    compute_sparsity_metrics=True,  # Enable sparsity metrics
    compute_variance_metrics=True,  # Enable variance metrics
)

# Create an activations store using the configuration and the dataset
activations_store = ActivationsStore.from_config(model, cfg, override_dataset=None)
activations_store.device = torch.device("mps")
sae_lens_sae.to(torch.device("mps"))

In [ ]:
run_evals(sae_lens_sae, activations_store, model, eval_cfg)